In [3]:
import pandas as pd

cleaned_path = r"C:\Users\bupes\OneDrive\Desktop\model\game_ml_platform\data\processed\games_cleaned.csv"

games = pd.read_csv(
    cleaned_path,
    low_memory=False
)

print("Shape:", games.shape)
print("Columns:", len(games.columns))
print("\nColumns:")
print(games.columns.tolist())

Shape: (125855, 39)
Columns: 39

Columns:
['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Required age', 'Price', 'Discount', 'DLC count', 'About the game', 'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 'Score rank', 'Achievements', 'Recommendations', 'Notes', 'Average playtime forever', 'Average playtime two weeks', 'Median playtime forever', 'Median playtime two weeks', 'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots']


In [4]:
# Verify The Witcher 3 data

witcher_check = games[
    games["Name"].astype(str).str.contains(
        "The Witcher 3: Wild Hunt",
        case=False,
        na=False,
        regex=False
    )
][[
    "AppID",
    "Name",
    "Release date",
    "Price",
    "Discount",
    "DLC count",
    "Metacritic score",
    "User score",
    "Positive",
    "Negative",
    "Peak CCU",
    "Developers",
    "Publishers",
    "Genres",
    "Tags"
]]

witcher_check

,AppID,Name,Release date,Price,Discount,DLC count,Metacritic score,User score,Positive,Negative,Peak CCU,Developers,Publishers,Genres,Tags
39800,292030,The Witcher 3: Wild Hunt,"May 18, 2015",3.99,90,22,93,0,802993,32356,15893,CD PROJEKT RED,CD PROJEKT RED,RPG,"Open World,RPG,Story Rich,Atmospheric,Mature,F..."


In [5]:
print(witcher_check.to_string(index=False))

 AppID                     Name Release date  Price  Discount  DLC count  Metacritic score  User score  Positive  Negative  Peak CCU     Developers     Publishers Genres                                                                                                                                                                                                        Tags
292030 The Witcher 3: Wild Hunt May 18, 2015   3.99        90         22                93           0    802993     32356     15893 CD PROJEKT RED CD PROJEKT RED    RPG Open World,RPG,Story Rich,Atmospheric,Mature,Fantasy,Adventure,Singleplayer,Nudity,Choices Matter,Great Soundtrack,Third Person,Medieval,Action,Multiple Endings,Action RPG,Dark Fantasy,Magic,Dark,Sandbox


In [6]:
important_columns = [
    "AppID",
    "Name",
    "Release date",
    "Estimated owners",
    "Peak CCU",
    "Price",
    "About the game",
    "Metacritic score",
    "User score",
    "Positive",
    "Negative",
    "Genres",
    "Tags",
    "Developers",
    "Publishers",
    "Categories"
]

games_ml = games[important_columns].copy()

print("Shape:", games_ml.shape)
print("\nColumns:")
print(games_ml.columns.tolist())

Shape: (125855, 16)

Columns:
['AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 'Price', 'About the game', 'Metacritic score', 'User score', 'Positive', 'Negative', 'Genres', 'Tags', 'Developers', 'Publishers', 'Categories']


In [7]:
# Remove games without a name
games_ml = games_ml.dropna(subset=["Name"])

# Remove duplicate game names
games_ml = games_ml.drop_duplicates(subset="Name", keep="first")

# Reset index
games_ml = games_ml.reset_index(drop=True)

# Text columns
text_columns = [
    "Genres",
    "Tags",
    "About the game",
    "Categories",
    "Developers",
    "Publishers"
]

for column in text_columns:
    games_ml[column] = (
        games_ml[column]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# Numeric columns
numeric_columns = [
    "Price",
    "Metacritic score",
    "User score",
    "Positive",
    "Negative",
    "Peak CCU"
]

for column in numeric_columns:
    games_ml[column] = pd.to_numeric(
        games_ml[column],
        errors="coerce"
    )

# Replace missing numeric values with 0
games_ml[numeric_columns] = games_ml[numeric_columns].fillna(0)

print("Shape after cleaning:", games_ml.shape)

print("\nMissing values:")
print(games_ml.isnull().sum())

print("\nSample:")
print(
    games_ml[
        ["Name", "Price", "Metacritic score",
         "Positive", "Negative", "Genres", "Tags"]
    ].head()
)

Shape after cleaning: (124644, 16)

Missing values:
AppID               0
Name                0
Release date        0
Estimated owners    0
Peak CCU            0
Price               0
About the game      0
Metacritic score    0
User score          0
Positive            0
Negative            0
Genres              0
Tags                0
Developers          0
Publishers          0
Categories          0
dtype: int64

Sample:
                                    Name  Price  Metacritic score  Positive  \
0             Black Dragon Mage Playtest   0.00                 0         0   
1  Supipara - Chapter 1 Spring Has Come!   5.24                 0       252   
2      Mystery Solitaire The Black Raven   4.99                 0        21   
3            버튜버 파라노이아 - Vtuber Paranoia   8.99                 0         0   
4                          Maze Quest VR   4.99                 0         0   

   Negative                   Genres  \
0         0                            
1         3        

In [8]:
# Replace commas with spaces so TF-IDF treats individual values as words
for column in [
    "Genres",
    "Tags",
    "Categories",
    "Developers",
    "Publishers"
]:
    games_ml[column] = games_ml[column].str.replace(
        ",", " ", regex=False
    )

# Create weighted recommendation features
games_ml["features"] = (
    games_ml["Genres"] + " " +
    games_ml["Genres"] + " " +
    games_ml["Tags"] + " " +
    games_ml["Tags"] + " " +
    games_ml["Tags"] + " " +
    games_ml["Categories"] + " " +
    games_ml["About the game"] + " " +
    games_ml["Developers"] + " " +
    games_ml["Publishers"]
)

# Remove extra whitespace
games_ml["features"] = (
    games_ml["features"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove games with no usable recommendation features
games_ml = games_ml[
    games_ml["features"] != ""
].copy()

games_ml = games_ml.reset_index(drop=True)

print("Shape:", games_ml.shape)

print("\nExample recommendation feature:")
print(games_ml.loc[0, "features"][:1000])

Shape: (116686, 17)

Example recommendation feature:
Adventure Adventure Adventure Visual Novel Anime Cute Adventure Visual Novel Anime Cute Adventure Visual Novel Anime Cute Single-player Steam Trading Cards Steam Cloud Family Sharing Springtime, April: when the cherry trees come into full bloom. The protagonist Yukinari Sanada has returned to his hometown in Kanagawa Prefecture, Kamakura City, for the first time in seven years, and is greeted by his older cousin Sakura Narumi (complete with maid outfit). He wanted to live in peace, but his life at the academy placed on the coastlands becomes bustling and brilliant while surrounded by a lineup of girls with booming personalities like Hotaru Amano, the sharp tongued, half-Japanese beauty, and Alice Kamishiro, the lazy witch who loves modern-science and mail-orders. Eventually, our protagonist is coaxed into joining the action committee for the academy's traditional beauty contest by his close friend. He interacts with the heroines part

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Convert game features into TF-IDF vectors
tfidf_weighted = TfidfVectorizer(
    stop_words="english",
    max_features=50000
)

tfidf_weighted_matrix = tfidf_weighted.fit_transform(
    games_ml["features"]
)

print("TF-IDF matrix shape:", tfidf_weighted_matrix.shape)
print("Number of features:", len(tfidf_weighted.get_feature_names_out()))

# Train nearest-neighbor similarity model
nn_weighted = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11
)

nn_weighted.fit(tfidf_weighted_matrix)

print("Recommendation model trained successfully!")

TF-IDF matrix shape: (116686, 50000)
Number of features: 50000
Recommendation model trained successfully!


In [10]:
def recommend_game(game_name, n=10):
    matches = games_ml[
        games_ml["Name"].str.contains(
            game_name,
            case=False,
            na=False,
            regex=False
        )
    ]

    if matches.empty:
        return f"No game found matching '{game_name}'."

    game_index = matches.index[0]

    distances, indices = nn_weighted.kneighbors(
        tfidf_weighted_matrix[game_index],
        n_neighbors=n + 1
    )

    recommended_indices = indices[0][1:]

    recommendations = games_ml.iloc[recommended_indices][
        ["AppID", "Name", "Genres", "Tags", "Price"]
    ].copy()

    recommendations["similarity"] = (
        1 - distances[0][1:]
    )

    return recommendations.reset_index(drop=True)

In [11]:
recommend_game("The Witcher 3: Wild Hunt", 10)

,AppID,Name,Genres,Tags,Price,similarity
0,1091500,Cyberpunk 2077,RPG,Cyberpunk Open World Nudity RPG Singleplayer S...,20.99,0.639809
1,20920,The Witcher 2: Assassins of Kings Enhanced Edi...,RPG,RPG Fantasy Mature Story Rich Choices Matter S...,2.99,0.549809
2,208730,Game of Thrones,Action RPG,RPG Fantasy Action Story Rich Third Person Cho...,14.99,0.510486
3,40300,Risen,RPG,RPG Open World Gothic Fantasy Adventure Single...,3.74,0.502926
4,253980,Enclave,Action RPG,RPG Action Fantasy Third Person Hack and Slash...,3.99,0.478987
5,20900,The Witcher: Enhanced Edition Director's Cut,Action RPG,RPG Fantasy Story Rich Mature Singleplayer Cho...,1.49,0.453491
6,2563960,Necroslayer,Action Adventure Indie RPG,Action Roguelike Open World Dungeon Crawler Da...,5.99,0.445727
7,606880,GreedFall,RPG,RPG Open World Character Customization Singlep...,3.49,0.439584
8,990080,Hogwarts Legacy,Action Adventure RPG,Magic Open World Fantasy Singleplayer Adventur...,5.99,0.436939
9,356190,Middle-earth™: Shadow of War™,Action Adventure RPG,Open World Action RPG Singleplayer Fantasy Adv...,4.99,0.436703


In [12]:
import numpy as np

def personalized_recommendations(liked_games, n=10):
    liked_indices = []

    for game_name in liked_games:
        matches = games_ml[
            games_ml["Name"].str.contains(
                game_name,
                case=False,
                na=False,
                regex=False
            )
        ]

        if not matches.empty:
            liked_indices.append(matches.index[0])

    if not liked_indices:
        return "None of the liked games were found."

    # Get TF-IDF vectors for liked games
    liked_vectors = tfidf_weighted_matrix[liked_indices]

    # Create user's preference profile
    user_profile = np.asarray(
        liked_vectors.mean(axis=0)
    )

    # Find games closest to the user profile
    candidate_count = min(
        len(games_ml),
        n + len(liked_indices) + 20
    )

    distances, indices = nn_weighted.kneighbors(
        user_profile,
        n_neighbors=candidate_count
    )

    recommendations = games_ml.iloc[indices[0]][
        ["AppID", "Name", "Genres", "Tags", "Price"]
    ].copy()

    recommendations["similarity"] = (
        1 - distances[0]
    )

    # Remove games the user already likes
    liked_appids = games_ml.iloc[
        liked_indices
    ]["AppID"].tolist()

    recommendations = recommendations[
        ~recommendations["AppID"].isin(liked_appids)
    ]

    return recommendations.head(n).reset_index(drop=True)

In [13]:
personalized_recommendations(
    [
        "The Witcher 3: Wild Hunt",
        "Cyberpunk 2077",
        "Hogwarts Legacy"
    ],
    10
)

,AppID,Name,Genres,Tags,Price,similarity
0,40300,Risen,RPG,RPG Open World Gothic Fantasy Adventure Single...,3.74,0.526439
1,20920,The Witcher 2: Assassins of Kings Enhanced Edi...,RPG,RPG Fantasy Mature Story Rich Choices Matter S...,2.99,0.496137
2,606880,GreedFall,RPG,RPG Open World Character Customization Singlep...,3.49,0.487066
3,338390,The Technomancer,Action RPG,RPG Action Sci-fi Open World Singleplayer Thir...,1.49,0.483712
4,208730,Game of Thrones,Action RPG,RPG Fantasy Action Story Rich Third Person Cho...,14.99,0.477470
5,1997660,GreedFall II: The Dying World,Action Adventure RPG Early Access,RPG Adventure Story Rich Fantasy Singleplayer ...,31.99,0.474543
6,253980,Enclave,Action RPG,RPG Action Fantasy Third Person Hack and Slash...,3.99,0.468126
7,243930,Bound By Flame,Action RPG,RPG Action Fantasy Singleplayer Adventure Hack...,0.76,0.468009
8,1681430,RoboCop: Rogue City,Action Adventure,FPS Action First-Person Singleplayer Shooter G...,15.99,0.466141
9,2563960,Necroslayer,Action Adventure Indie RPG,Action Roguelike Open World Dungeon Crawler Da...,5.99,0.445234


In [14]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Create a separate dataset for price prediction
price_data = games_ml.copy()

# Total number of reviews
price_data["total_reviews"] = (
    price_data["Positive"] +
    price_data["Negative"]
)

# Positive review ratio
price_data["positive_ratio"] = np.where(
    price_data["total_reviews"] > 0,
    price_data["Positive"] / price_data["total_reviews"],
    0
)

# Convert release date to datetime
price_data["Release date"] = pd.to_datetime(
    price_data["Release date"],
    format="mixed",
    errors="coerce"
)

# Extract release year
price_data["Release year"] = (
    price_data["Release date"].dt.year
)

# Use 2026 as the current year
current_year = 2026

price_data["Release year"] = (
    price_data["Release year"]
    .fillna(current_year)
)

# Calculate game age
price_data["game_age"] = (
    current_year - price_data["Release year"]
).clip(lower=0)

# Features for price prediction
feature_columns = [
    "Metacritic score",
    "User score",
    "Positive",
    "Negative",
    "Peak CCU",
    "Release year",
    "total_reviews",
    "positive_ratio",
    "game_age"
]

X = price_data[feature_columns].copy()

# Target
y = pd.to_numeric(
    price_data["Price"],
    errors="coerce"
).fillna(0)

# Handle infinite/missing values
X = X.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(feature_columns)

print("\nSample X:")
print(X.head())

print("\nSample y:")
print(y.head())

X shape: (116686, 9)
y shape: (116686,)

Features:
['Metacritic score', 'User score', 'Positive', 'Negative', 'Peak CCU', 'Release year', 'total_reviews', 'positive_ratio', 'game_age']

Sample X:
   Metacritic score  User score  Positive  Negative  Peak CCU  Release year  \
0                 0           0       252         3         0          2016   
1                 0           0        21         3         0          2019   
2                 0           0         0         0         1          2024   
3                 0           0         0         0         0          2025   
4                 0           0         0         0         0          2023   

   total_reviews  positive_ratio  game_age  
0            255        0.988235        10  
1             24        0.875000         7  
2              0        0.000000         2  
3              0        0.000000         1  
4              0        0.000000         3  

Sample y:
0     5.24
1     4.99
2     8.99
3     4.99
4   

In [15]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

# Create Random Forest model
price_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train the model
price_model.fit(X_train, y_train)

print("\nPrice prediction model trained successfully!")

Training samples: 93348
Testing samples: 23338

Price prediction model trained successfully!


In [16]:
# Make predictions on test data
y_pred = price_model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Price Prediction Results")
print("=" * 35)

print(f"MAE  : ${mae:.2f}")
print(f"RMSE : ${rmse:.2f}")
print(f"R²   : {r2:.4f}")

Price Prediction Results
MAE  : $4.86
RMSE : $15.51
R²   : -0.0297


In [17]:
print("Price statistics")
print("=" * 35)

print(y.describe())

print("\nPrice percentiles:")
print(y.quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nNumber of free games:", (y == 0).sum())
print("Percentage free:", f"{(y == 0).mean() * 100:.2f}%")

print("\nMost common prices:")
print(y.value_counts().head(15))

Price statistics
count    116686.000000
mean          5.148421
std          12.884677
min           0.000000
25%           0.890000
50%           2.790000
75%           5.990000
max         999.980000
Name: Price, dtype: float64

Price percentiles:
0.25     0.89
0.50     2.79
0.75     5.99
0.90    10.79
0.95    14.99
0.99    29.99
Name: Price, dtype: float64

Number of free games: 18435
Percentage free: 15.80%

Most common prices:
Price
0.00     18435
0.99      8879
4.99      8140
1.99      6643
2.99      6399
9.99      4698
3.99      4641
0.49      3596
5.99      3158
7.99      2257
2.49      2250
6.99      2166
1.49      2160
14.99     2130
8.99      1434
Name: count, dtype: int64


In [18]:
# Create a modeling dataset without extreme price outliers
price_model_data = price_data[
    price_data["Price"] <= 59.99
].copy()

print("Original games:", len(price_data))
print("Games used for price model:", len(price_model_data))
print(
    "Removed:",
    len(price_data) - len(price_model_data)
)

print("\nNew price statistics:")
print(price_model_data["Price"].describe())

Original games: 116686
Games used for price model: 116283
Removed: 403

New price statistics:
count    116283.000000
mean          4.549185
std           5.946902
min           0.000000
25%           0.890000
50%           2.790000
75%           5.990000
max          59.990000
Name: Price, dtype: float64


In [19]:
# Build features and target from the filtered dataset
X_price = price_model_data[feature_columns].copy()

y_price = pd.to_numeric(
    price_model_data["Price"],
    errors="coerce"
).fillna(0)

# Handle any remaining invalid values
X_price = X_price.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

print("X shape:", X_price.shape)
print("y shape:", y_price.shape)

# Train/test split
X_train_price, X_test_price, y_train_price, y_test_price = train_test_split(
    X_price,
    y_price,
    test_size=0.2,
    random_state=42
)

print("\nTraining samples:", len(X_train_price))
print("Testing samples:", len(X_test_price))

# Train new Random Forest
price_model_v2 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

price_model_v2.fit(
    X_train_price,
    y_train_price
)

print("\nPrice model v2 trained successfully!")

X shape: (116283, 9)
y shape: (116283,)

Training samples: 93026
Testing samples: 23257

Price model v2 trained successfully!


In [20]:
# Predictions from Model v2
y_pred_v2 = price_model_v2.predict(X_test_price)

# Evaluation metrics
mae_v2 = mean_absolute_error(
    y_test_price,
    y_pred_v2
)

rmse_v2 = np.sqrt(
    mean_squared_error(
        y_test_price,
        y_pred_v2
    )
)

r2_v2 = r2_score(
    y_test_price,
    y_pred_v2
)

print("Price Prediction Model v2")
print("=" * 35)

print(f"MAE  : ${mae_v2:.2f}")
print(f"RMSE : ${rmse_v2:.2f}")
print(f"R²   : {r2_v2:.4f}")

print("\nComparison with Model v1")
print("=" * 35)

print(f"Model v1 MAE  : ${mae:.2f}")
print(f"Model v1 RMSE : ${rmse:.2f}")
print(f"Model v1 R²   : {r2:.4f}")

print(f"\nModel v2 MAE  : ${mae_v2:.2f}")
print(f"Model v2 RMSE : ${rmse_v2:.2f}")
print(f"Model v2 R²   : {r2_v2:.4f}")

Price Prediction Model v2
MAE  : $3.81
RMSE : $5.95
R²   : 0.0164

Comparison with Model v1
Model v1 MAE  : $4.86
Model v1 RMSE : $15.51
Model v1 R²   : -0.0297

Model v2 MAE  : $3.81
Model v2 RMSE : $5.95
Model v2 R²   : 0.0164


In [21]:
feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": price_model_v2.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print(feature_importance)

            Feature  Importance
0    positive_ratio    0.218675
1     total_reviews    0.171869
2          Peak CCU    0.153250
3          Positive    0.138985
4          Negative    0.129925
5          game_age    0.085250
6      Release year    0.067264
7  Metacritic score    0.034130
8        User score    0.000651


In [22]:
categorical_columns = [
    "Genres",
    "Tags",
    "Categories",
    "Developers",
    "Publishers"
]

print("Unique values in categorical features")
print("=" * 45)

for column in categorical_columns:
    print(
        f"{column:15} : "
        f"{games_ml[column].nunique():,} unique values"
    )

Unique values in categorical features
Genres          : 2,910 unique values
Tags            : 76,461 unique values
Categories      : 13,990 unique values
Developers      : 72,231 unique values
Publishers      : 64,042 unique values


In [23]:
# Start with the existing price-model dataset
price_model_data_v3 = price_model_data.copy()

# Number of entries in each categorical field
price_model_data_v3["genre_count"] = (
    price_model_data_v3["Genres"]
    .str.split()
    .str.len()
)

price_model_data_v3["tag_count"] = (
    price_model_data_v3["Tags"]
    .str.split()
    .str.len()
)

price_model_data_v3["category_count"] = (
    price_model_data_v3["Categories"]
    .str.split()
    .str.len()
)

price_model_data_v3["developer_count"] = (
    price_model_data_v3["Developers"]
    .str.split()
    .str.len()
)

price_model_data_v3["publisher_count"] = (
    price_model_data_v3["Publishers"]
    .str.split()
    .str.len()
)

# Frequency of developer/publisher names
developer_frequency = (
    games_ml["Developers"]
    .value_counts()
)

publisher_frequency = (
    games_ml["Publishers"]
    .value_counts()
)

price_model_data_v3["developer_frequency"] = (
    price_model_data_v3["Developers"]
    .map(developer_frequency)
    .fillna(0)
)

price_model_data_v3["publisher_frequency"] = (
    price_model_data_v3["Publishers"]
    .map(publisher_frequency)
    .fillna(0)
)

print("New features created successfully!")

print("\nNew feature samples:")
print(
    price_model_data_v3[
        [
            "Name",
            "genre_count",
            "tag_count",
            "category_count",
            "developer_count",
            "publisher_count",
            "developer_frequency",
            "publisher_frequency"
        ]
    ].head()
)

New features created successfully!

New feature samples:
                                    Name  genre_count  tag_count  \
0  Supipara - Chapter 1 Spring Has Come!            1          5   
1      Mystery Solitaire The Black Raven            1         19   
2            버튜버 파라노이아 - Vtuber Paranoia            3          0   
3                          Maze Quest VR            3          0   
4                               Agony VR            2          0   

   category_count  developer_count  publisher_count  developer_frequency  \
0               8                1                1                    8   
1               3                2                1                   70   
2               5                1                1                   13   
3               7                3                3                    2   
4               8                1                3                    5   

   publisher_frequency  
0                   77  
1                  267  
2 

In [24]:
v3_feature_columns = feature_columns + [
    "genre_count",
    "tag_count",
    "category_count",
    "developer_count",
    "publisher_count"
]

X_v3 = price_model_data_v3[v3_feature_columns].copy()

y_v3 = pd.to_numeric(
    price_model_data_v3["Price"],
    errors="coerce"
).fillna(0)

# Handle invalid values
X_v3 = X_v3.replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

print("X_v3 shape:", X_v3.shape)
print("y_v3 shape:", y_v3.shape)

print("\nModel v3 features:")
print(v3_feature_columns)

print("\nSample:")
print(X_v3.head())

X_v3 shape: (116283, 14)
y_v3 shape: (116283,)

Model v3 features:
['Metacritic score', 'User score', 'Positive', 'Negative', 'Peak CCU', 'Release year', 'total_reviews', 'positive_ratio', 'game_age', 'genre_count', 'tag_count', 'category_count', 'developer_count', 'publisher_count']

Sample:
   Metacritic score  User score  Positive  Negative  Peak CCU  Release year  \
0                 0           0       252         3         0          2016   
1                 0           0        21         3         0          2019   
2                 0           0         0         0         1          2024   
3                 0           0         0         0         0          2025   
4                 0           0         0         0         0          2023   

   total_reviews  positive_ratio  game_age  genre_count  tag_count  \
0            255        0.988235        10            1          5   
1             24        0.875000         7            1         19   
2              0     

In [25]:
# Train/test split
X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(
    X_v3,
    y_v3,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train_v3))
print("Testing samples:", len(X_test_v3))

# Random Forest v3
price_model_v3 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

price_model_v3.fit(
    X_train_v3,
    y_train_v3
)

print("\nPrice model v3 trained successfully!")

Training samples: 93026
Testing samples: 23257

Price model v3 trained successfully!


In [26]:
# Predictions
y_pred_v3 = price_model_v3.predict(X_test_v3)

# Metrics
mae_v3 = mean_absolute_error(
    y_test_v3,
    y_pred_v3
)

rmse_v3 = np.sqrt(
    mean_squared_error(
        y_test_v3,
        y_pred_v3
    )
)

r2_v3 = r2_score(
    y_test_v3,
    y_pred_v3
)

print("Price Prediction Model v3")
print("=" * 35)

print(f"MAE  : ${mae_v3:.2f}")
print(f"RMSE : ${rmse_v3:.2f}")
print(f"R²   : {r2_v3:.4f}")

print("\nModel Comparison")
print("=" * 35)

print(f"Model v1 → MAE: ${mae:.2f}, RMSE: ${rmse:.2f}, R²: {r2:.4f}")
print(f"Model v2 → MAE: ${mae_v2:.2f}, RMSE: ${rmse_v2:.2f}, R²: {r2_v2:.4f}")
print(f"Model v3 → MAE: ${mae_v3:.2f}, RMSE: ${rmse_v3:.2f}, R²: {r2_v3:.4f}")

Price Prediction Model v3
MAE  : $3.55
RMSE : $5.59
R²   : 0.1308

Model Comparison
Model v1 → MAE: $4.86, RMSE: $15.51, R²: -0.0297
Model v2 → MAE: $3.81, RMSE: $5.95, R²: 0.0164
Model v3 → MAE: $3.55, RMSE: $5.59, R²: 0.1308


In [27]:
# Compare actual vs predicted prices
comparison = pd.DataFrame({
    "Actual Price": y_test_v3.values,
    "Predicted Price": y_pred_v3
})

comparison["Absolute Error"] = (
    comparison["Actual Price"] -
    comparison["Predicted Price"]
).abs()

print(comparison.head(20).to_string(index=False))

 Actual Price  Predicted Price  Absolute Error
         5.99         1.321007        4.668993
         1.79         3.072650        1.282650
         0.00         3.915523        3.915523
        23.99         5.362353       18.627647
         0.89         6.836100        5.946100
         1.99         1.674004        0.315996
         3.99         6.951700        2.961700
         3.99         3.116151        0.873849
         2.49         1.443500        1.046500
         3.99         1.463403        2.526597
         0.00         0.538800        0.538800
        19.99         5.427631       14.562369
         8.39         5.003600        3.386400
         2.49         3.022350        0.532350
         3.19         4.273474        1.083474
         1.99         5.155133        3.165133
         0.00         0.000000        0.000000
         4.79         4.526800        0.263200
        10.99         3.964024        7.025976
         0.49         1.179200        0.689200


In [28]:
# Log-transform the target price
y_v4 = np.log1p(y_v3)

print("Original price examples:")
print(y_v3.head(10).values)

print("\nLog-transformed examples:")
print(y_v4.head(10).values)

print("\nOriginal target statistics:")
print(y_v3.describe())

print("\nLog target statistics:")
print(y_v4.describe())

Original price examples:
[ 5.24  4.99  8.99  4.99 13.99 35.99  0.99  0.99  0.59  0.99]

Log-transformed examples:
[1.83098018 1.79009141 2.30158459 1.79009141 2.70738331 3.61064761
 0.68813464 0.68813464 0.46373402 0.68813464]

Original target statistics:
count    116283.000000
mean          4.549185
std           5.946902
min           0.000000
25%           0.890000
50%           2.790000
75%           5.990000
max          59.990000
Name: Price, dtype: float64

Log target statistics:
count    116283.000000
mean          1.305218
std           0.893345
min           0.000000
25%           0.636577
50%           1.332366
75%           1.944481
max           4.110710
Name: Price, dtype: float64


In [29]:
# Train Random Forest using log-transformed prices
price_model_v4 = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

price_model_v4.fit(
    X_train_v3,
    np.log1p(y_train_v3)
)

print("Price model v4 trained successfully!")

Price model v4 trained successfully!


In [30]:
# Predict in log scale
y_pred_v4_log = price_model_v4.predict(X_test_v3)

# Convert predictions back to dollar prices
y_pred_v4 = np.expm1(y_pred_v4_log)

# Prevent tiny negative floating-point values
y_pred_v4 = np.maximum(y_pred_v4, 0)

# Calculate metrics
mae_v4 = mean_absolute_error(
    y_test_v3,
    y_pred_v4
)

rmse_v4 = np.sqrt(
    mean_squared_error(
        y_test_v3,
        y_pred_v4
    )
)

r2_v4 = r2_score(
    y_test_v3,
    y_pred_v4
)

print("Price Prediction Model v4")
print("=" * 35)

print(f"MAE  : ${mae_v4:.2f}")
print(f"RMSE : ${rmse_v4:.2f}")
print(f"R²   : {r2_v4:.4f}")

print("\nModel Comparison")
print("=" * 35)

print(f"Model v2 → MAE: ${mae_v2:.2f}, RMSE: ${rmse_v2:.2f}, R²: {r2_v2:.4f}")
print(f"Model v3 → MAE: ${mae_v3:.2f}, RMSE: ${rmse_v3:.2f}, R²: {r2_v3:.4f}")
print(f"Model v4 → MAE: ${mae_v4:.2f}, RMSE: ${rmse_v4:.2f}, R²: {r2_v4:.4f}")

Price Prediction Model v4
MAE  : $3.19
RMSE : $5.70
R²   : 0.0965

Model Comparison
Model v2 → MAE: $3.81, RMSE: $5.95, R²: 0.0164
Model v3 → MAE: $3.55, RMSE: $5.59, R²: 0.1308
Model v4 → MAE: $3.19, RMSE: $5.70, R²: 0.0965


In [31]:
comparison_v4 = pd.DataFrame({
    "Actual Price": y_test_v3.values,
    "Predicted Price": y_pred_v4
})

comparison_v4["Absolute Error"] = (
    comparison_v4["Actual Price"] -
    comparison_v4["Predicted Price"]
).abs()

print(
    comparison_v4.head(20).to_string(index=False)
)

 Actual Price  Predicted Price  Absolute Error
         5.99         1.245461        4.744539
         1.79         2.115009        0.325009
         0.00         2.101182        2.101182
        23.99         4.445039       19.544961
         0.89         4.524243        3.634243
         1.99         1.194695        0.795305
         3.99         5.769741        1.779741
         3.99         1.847163        2.142837
         2.49         1.244795        1.245205
         3.99         1.126727        2.863273
         0.00         0.498731        0.498731
        19.99         4.443400       15.546600
         8.39         3.088889        5.301111
         2.49         2.181190        0.308810
         3.19         3.418506        0.228506
         1.99         4.274702        2.284702
         0.00         0.000000        0.000000
         4.79         3.468799        1.321201
        10.99         2.627845        8.362155
         0.49         0.568819        0.078819


In [32]:
feature_importance_v3 = pd.DataFrame({
    "Feature": v3_feature_columns,
    "Importance": price_model_v3.feature_importances_
})

feature_importance_v3 = feature_importance_v3.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

print(feature_importance_v3.to_string(index=False))

         Feature  Importance
  category_count    0.144182
       tag_count    0.111564
     genre_count    0.102125
  positive_ratio    0.098503
   total_reviews    0.079028
        Peak CCU    0.075595
        Positive    0.069368
 developer_count    0.066289
        Negative    0.066128
 publisher_count    0.064762
        game_age    0.057361
    Release year    0.050663
Metacritic score    0.014180
      User score    0.000251


In [33]:
import os
import joblib

models_path = r"C:\Users\bupes\OneDrive\Desktop\model\game_ml_platform\models"

os.makedirs(models_path, exist_ok=True)

# -----------------------------
# Recommendation model
# -----------------------------

joblib.dump(
    tfidf_weighted,
    os.path.join(models_path, "tfidf_vectorizer.pkl")
)

joblib.dump(
    nn_weighted,
    os.path.join(models_path, "recommendation_model.pkl")
)

games_ml.to_pickle(
    os.path.join(models_path, "games_ml.pkl")
)

# -----------------------------
# Price prediction model
# -----------------------------

joblib.dump(
    price_model_v4,
    os.path.join(models_path, "price_model.pkl")
)

joblib.dump(
    v3_feature_columns,
    os.path.join(models_path, "price_features.pkl")
)

print("All models saved successfully!\n")

print("Saved files:")

for filename in [
    "tfidf_vectorizer.pkl",
    "recommendation_model.pkl",
    "games_ml.pkl",
    "price_model.pkl",
    "price_features.pkl"
]:
    filepath = os.path.join(models_path, filename)

    print(
        f"{filename:30} "
        f"{os.path.getsize(filepath) / (1024 * 1024):.2f} MB"
    )

All models saved successfully!

Saved files:
tfidf_vectorizer.pkl           1.79 MB
recommendation_model.pkl       143.93 MB
games_ml.pkl                   382.10 MB
price_model.pkl                1088.00 MB
price_features.pkl             0.00 MB
